# Week 07 — Home exercise 3: Two awkward files

**Solution proposal.**

Two files that do not open correctly by default, and the arguments each one needs.

In [1]:
import pandas as pd

## 1. `survey_data.csv` — the wrong separator

The default read gives **one** column whose name is the entire header line. That is the signature of
a wrong separator, and it is worth recognizing on sight.

In [2]:
wrong = pd.read_csv("../data/survey_data.csv")

print("default read:", wrong.shape)
print("its one column:", list(wrong.columns))

default read: (2884, 1)
its one column: ['respondentID:age:hourly_earnings:years_schooling:sex:sector:experience']


Opening the file in a text editor shows the separator is a colon — unusual, but perfectly legal.

In [3]:
survey = pd.read_csv("../data/survey_data.csv", sep=":")

print("with sep=':':", survey.shape)
print(list(survey.columns))
survey.head(3)

with sep=':': (2884, 7)
['respondentID', 'age', 'hourly_earnings', 'years_schooling', 'sex', 'sector', 'experience']


,respondentID,age,hourly_earnings,years_schooling,sex,sector,experience
0,1,50,109.357913,9,male,private,30
1,2,50,267.671518,15,male,public,26
2,3,46,193.239479,15,female,public,29


## 2. `eurostat.xlsx` — three things wrong at once

| argument | fixes |
|---|---|
| `sheet_name` | the workbook has three sheets and the data is on the third |
| `skiprows` | nine rows of extraction metadata sit above the real header |
| `usecols` | every second column is an empty flag column |

Each argument fixes exactly one problem, and you find them by saying what is wrong **in words** first
and then searching the documentation.

In [4]:
print("sheets:", pd.ExcelFile("../data/eurostat.xlsx").sheet_names)

sheets: ['Summary', 'Structure', 'Sheet 1']


The country names are at position 0 and the years are at the odd positions after it, so the
columns we want are 0 followed by 1, 3, 5 and so on.

In [5]:
wanted = [0] + list(range(1, 47, 2))

electricity = pd.read_excel(
    "../data/eurostat.xlsx",
    sheet_name="Sheet 1",
    skiprows=9,
    usecols=wanted,
)

print("electricity:", electricity.shape)
print(list(electricity.columns)[:6], "...")
electricity.iloc[:6, :4]

electricity: (49, 24)
['TIME', '2001', '2002', '2003', '2004', '2005'] ...


,TIME,2001,2002,2003
0,GEO (Labels),NaN,NaN,NaN
1,European Union - 27 countries (from 2020),2353732.207,2383768.335,2449917.149
2,Euro area – 20 countries (from 2023),1922726.207,1955922.335,2013462.149
3,Belgium,79816,80438,82065
4,Bulgaria,25776,25287,26425
5,Czechia,53775,53669,54807


## Which rows are not countries?

The first two data rows are `European Union - 27 countries (from 2020)` and
`Euro area - 20 countries (from 2023)`. Neither is a country; both are sums over countries that also
appear individually further down. Add the column up and you count Belgium three times.

The row above them is worse: `GEO (Labels)` is a leftover header fragment with no data at all, which
survived because it sat below the row we told `skiprows` to stop at.

**How would you recognize them without being told?** Three signals, in order of reliability:

1. **They have no country code.** This file does not have a code column, but most sources do, and
   aggregates are the rows where it is blank or is something like `EU27` rather than a three-letter
   ISO code. This is the reliable one, and it is the check the emissions data lets you make.
2. **Their values are far larger than any individual row.** An aggregate is a sum, so it sits above
   everything it contains. Sorting by the largest value brings them all to the top at once.
3. **Their names say so**, in words like "total", "area", "countries", "income" or "World". Readable,
   but the least reliable: it depends on the publisher's naming and will not survive translation.

### What this notebook does NOT do

- It reads the Eurostat file but does not finish cleaning it. The first column is still called `TIME`
  even though it holds country names, the first row is still the `GEO (Labels)` fragment, the
  aggregate rows are still there, and the years are still spread across 23 columns instead of being a
  column of their own. Turning that into a usable table is reshaping work that comes later.
- It does not check what the numbers mean. Eurostat writes some cells as `:` for "not available", and
  depending on the export those can arrive as text rather than as `NaN`, which would quietly make a
  whole column non-numeric. Always check `.dtypes` after reading a file like this, not just `.shape`.